In [8]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


In [38]:
import pandas as pd
import numpy as np

#carregando o dataset
file_inmet = '/content/drive/MyDrive/Pivic/dados/Dados/Institudo de Metereologia/dados 2025/INMET_NE_PB_A313_CAMPINA GRANDE_01-01-2025_A_31-10-2025.CSV'

df_clima = pd.read_csv(file_inmet,
                       sep=';',
                       decimal=',',
                       skiprows=8,
                       encoding='latin1')

# selecionando as colunas que serão úteis
colunas_map = {
    'Data': 'DATA',
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)': 'CHUVA_TOTAL',
    'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)': 'TEMP_MEDIA',
    'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)': 'TEMP_MAX',
    'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)': 'TEMP_MIN',
    'UMIDADE RELATIVA DO AR, HORARIA (%)': 'UMIDADE'
}
df_clima = df_clima.rename(columns=colunas_map)[colunas_map.values()]


#tratando irregularidade

df_clima['DATA'] = pd.to_datetime(df_clima['DATA'], format='%Y/%m/%d')
df_clima = df_clima.replace(-9999, np.nan)

#transformando o valor em datetime para semana, pq coloquei os valores na tabela do dataset da epidemiologia em semanas
df_clima['SEMANA'] = df_clima['DATA'].dt.isocalendar().week


#salvando os parâmetros por semana
df_clima_semanal = df_clima.groupby('SEMANA').agg({
    'CHUVA_TOTAL': 'sum',
    'TEMP_MEDIA': 'mean',
    'TEMP_MAX': 'max',
    'TEMP_MIN': 'min',
    'UMIDADE': 'mean'
}).reset_index()

df_clima_semanal = df_clima_semanal.round(2)



df_dengue = pd.read_csv('/content/drive/MyDrive/Pivic/tratamento dados/dados tratados/epidemiologia/dengue2025_tratado.csv')

#transformando os valores em inteiro para que o cruzamento das bases não dê bronca
df_dengue['SEMANA EPDM'] = pd.to_numeric(df_dengue['SEMANA EPDM'], errors='coerce')

#criando uma coluna para agregar os casos em quantidade por semana
df_casos_semanal = df_dengue.groupby('SEMANA EPDM').size().reset_index(name='TOTAL_CASOS')
df_casos_semanal = df_casos_semanal.rename(columns={'SEMANA EPDM': 'SEMANA'})



#fazendo o merge propriamente dito, utilizando o inner, serve pra utilizar apenas colunas que estão presente nas duas tabelas
df_final = pd.merge(df_clima_semanal, df_casos_semanal, on='SEMANA', how='inner')

#salvando
#df_final.to_csv('dataset_clima/semana_epdm.csv', index=False)

df_final.head(10)



#dá pra replicar esse código aqui no resto

,SEMANA,CHUVA_TOTAL,TEMP_MEDIA,TEMP_MAX,TEMP_MIN,UMIDADE,TOTAL_CASOS
0,1,0.0,26.01,33.9,21.8,75.99,2
1,2,2.8,25.98,33.8,21.6,77.49,1
2,3,14.6,24.94,32.8,21.5,81.50,1
3,4,17.2,25.28,32.8,21.2,78.98,8
4,5,35.6,24.29,31.3,19.4,85.09,4
5,6,57.0,23.41,31.7,20.2,88.05,5
6,7,2.4,24.70,31.1,20.9,81.16,2
7,8,0.8,25.14,32.3,20.9,79.82,6
8,9,7.8,24.55,32.7,20.7,83.13,5
9,11,35.0,24.64,31.2,20.7,84.79,4


In [41]:
df_final.to_csv('dataset_clima_semana_epdm.csv', index=False)

In [40]:
df_clima_semanal.to_csv('dataset_clima_2025_tratato.csv',index=False)